# Biomedical Large-Image Segmentation Inference

This notebook uses MMSeg's ViT-S/16 + UPerNet for **slide inference** on large images, suitable for CLSM and other biomedical image segmentation.

**Before running:**
1. Place this notebook in the project `dense_prediction` (or `github_release_test`) folder and run from that directory.
2. Install dependencies: `pip install torch torchvision mmcv mmengine` (see README for versions).
3. Download the segmentation checkpoint (e.g. `iter_8000.pth`) and set its path below.

## 1. Environment and paths

In [1]:
import os
import sys

# 将当前目录（dense_prediction）加入路径，保证能 import mmseg
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)

Project root: /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test


## 2. Config paths (edit as needed)

In [2]:
# Config file path (usually no need to change)
CONFIG_PATH = os.path.join(project_root, "configs", "our", "small_upernet_test.py")

# Segmentation checkpoint path (set to your downloaded checkpoint)
CHECKPOINT_PATH = os.path.join(project_root, "checkpoints", "iter_8000.pth")

# Input: path to a single image or a folder of images
INPUT_IMAGE_OR_DIR = os.path.join(project_root, "my_images")

# Output directory for segmentation and visualizations
OUTPUT_DIR = os.path.join(project_root, "outputs")

# Device: use "cpu" if no GPU
DEVICE = "cuda:0" if __import__("torch").cuda.is_available() else "cpu"
print("CONFIG:", CONFIG_PATH)
print("CHECKPOINT:", CHECKPOINT_PATH)
print("INPUT:", INPUT_IMAGE_OR_DIR)
print("OUTPUT:", OUTPUT_DIR)
print("DEVICE:", DEVICE)

CONFIG: /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/configs/our/small_upernet_test.py
CHECKPOINT: /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/checkpoints/iter_8000.pth
INPUT: /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/my_images
OUTPUT: /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/outputs
DEVICE: cuda:0


## 3. Run inference with test_dataloader (same as test.sh)

Images are copied to a temp directory with placeholder annotations; Runner + test_dataloader runs the test pipeline. Results and visualizations are written to OUTPUT_DIR.

In [3]:
import glob
import shutil
import numpy as np
from PIL import Image
from mmengine.config import Config
from mmengine.runner import Runner

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) Collect image paths to run inference on
if os.path.isfile(INPUT_IMAGE_OR_DIR):
    image_paths = [INPUT_IMAGE_OR_DIR]
else:
    exts = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"]
    image_paths = []
    for ext in exts:
        image_paths.extend(glob.glob(os.path.join(INPUT_IMAGE_OR_DIR, ext)))
    image_paths = sorted(image_paths)

if not image_paths:
    print("No images found. Check INPUT_IMAGE_OR_DIR.")
else:
    print(f"共 {len(image_paths)} 张图片。")

    # 2) Prepare temp dataset dir (matches test_dataloader data_prefix in config)
    temp_data_root = os.path.join(project_root, "temp_infer_data")
    img_dir = os.path.join(temp_data_root, "images", "validation")
    ann_dir = os.path.join(temp_data_root, "annotations", "validation")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(ann_dir, exist_ok=True)

    for img_path in image_paths:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        dst_img = os.path.join(img_dir, base_name + ext)
        if os.path.abspath(img_path) != os.path.abspath(dst_img):
            shutil.copy2(img_path, dst_img)
        # Placeholder annotation: same-name .png, all zeros (background); required by LoadAnnotations
        try:
            im = np.array(Image.open(img_path))
            h, w = im.shape[:2]
        except Exception:
            h, w = 512, 512
        seg = np.zeros((h, w), dtype=np.uint8)
        Image.fromarray(seg).save(os.path.join(ann_dir, base_name + ".png"))

    # 3) 加载 config，覆盖 data_root、checkpoint、输出路径
    cfg = Config.fromfile(CONFIG_PATH)
    cfg.launcher = "none"
    cfg.load_from = CHECKPOINT_PATH
    cfg.work_dir = OUTPUT_DIR
    cfg.test_evaluator["output_dir"] = OUTPUT_DIR
    cfg.test_evaluator["keep_results"] = True
    cfg.test_dataloader.dataset.data_root = temp_data_root

    # 4) Enable visualization and set save directory
    if "visualization" in cfg.default_hooks:
        cfg.default_hooks["visualization"]["draw"] = True
        cfg.visualizer["save_dir"] = OUTPUT_DIR

    # 5) Build Runner and run test (same as tools/test.sh)
    runner = Runner.from_cfg(cfg)
    runner.test()

    print("Done. Segmentation and visualizations saved to:", OUTPUT_DIR)

共 2 张图片。
03/01 18:52:13 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1801534778
    GPU 0,1,2,3,4,5,6,7: NVIDIA RTX 6000 Ada Generation
    CUDA_HOME: /homes/yunhengwu/mambaforge/envs/seg
    NVCC: Cuda compilation tools, release 12.6, V12.6.20
    GCC: gcc (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
    PyTorch: 2.2.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.3.2 (Git Hash 2dc95a2ad0841e29db8b22fbccaf3e5da7992b01)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  -

/homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/mmseg/models/decode_heads/decode_head.py:122: UserWarning: For binary segmentation, we suggest using`out_channels = 1` to define the outputchannels of segmentor, and use `threshold`to convert `seg_logits` into a predictionapplying a threshold
  warnings.warn('For binary segmentation, we suggest using'
/homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/mmseg/models/builder.py:36: UserWarning: ``build_loss`` would be deprecated soon, please use ``mmseg.registry.MODELS.build()`` 
  warnings.warn('``build_loss`` would be deprecated soon, please use '
/homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/mmseg/models/losses/cross_entropy_loss.py:253: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


03/01 18:52:17 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
03/01 18:52:17 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                

/homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/mmseg/datasets/transforms/loading.py:83: UserWarning: `reduce_zero_label` will be deprecated, if you would like to ignore the zero label, please set `reduce_zero_label=True` when dataset initialized
  warnings.warn('`reduce_zero_label` will be deprecated, '


03/01 18:52:18 - mmengine - WARNING - The prefix is not set in metric class IoUMetric.
Loads checkpoint by local backend from path: /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/checkpoints/iter_8000.pth
03/01 18:52:19 - mmengine - INFO - Load checkpoint from /homes/yunhengwu/code/D1/CLSM_FM/dense_prediction/github_release_test/checkpoints/iter_8000.pth
[Slide Inference] Batch Size: 4, Elapsed: 0.6908s, FPS (batch): 5.79, FPS (single img): 5.79
[Slide Inference] Batch Size: 4, Elapsed: 0.1762s, FPS (batch): 22.70, FPS (single img): 22.70
[Slide Inference] Batch Size: 4, Elapsed: 0.1934s, FPS (batch): 20.68, FPS (single img): 20.68
[Slide Inference] Batch Size: 4, Elapsed: 0.1759s, FPS (batch): 22.74, FPS (single img): 22.74
[Slide Inference] Batch Size: 4, Elapsed: 0.1760s, FPS (batch): 22.73, FPS (single img): 22.73
[Slide Inference] Batch Size: 4, Elapsed: 0.1719s, FPS (batch): 23.26, FPS (single img): 23.26
[Slide Inference] Batch Size: 4, Elapsed: 0.1721s, F